# Bài 31: Nền tảng mạng nơ-ron (Neural Network Fundamentals)

Bài 30 đã chọn thủ công một hướng chiếu $w$ (hiệu của các trung bình lớp) để tách hai lớp. Cách đó hoạt động được, nhưng đòi hỏi con người phải nhận ra một mẹo suy nghiệm tốt. Bài này thay con người bằng **hạ gradient (gradient descent)**: một quy trình tự động *học* $w$ và $b$ trực tiếp từ dữ liệu bằng cách liên tục điều chỉnh nhẹ chúng để giảm một hàm mất mát (loss). Công thức &mdash; lượt xuôi (forward pass), hàm mất mát, lượt ngược (backward pass), cập nhật &mdash; là toàn bộ vòng lặp huấn luyện đằng sau mọi mạng nơ-ron trong khóa học này, bất kể lớn đến đâu.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

## Hạ gradient: dùng độ dốc để tìm cực tiểu

Huấn luyện một mô hình nghĩa là cực tiểu hóa một hàm mất mát theo các tham số của nó. Trước khi mở rộng khái niệm sang nhiều tham số, hãy xem ý tưởng ở dạng đơn giản nhất: một hàm 1D của một số $x$ duy nhất.

Tại bất kỳ điểm $x$ nào, đạo hàm $f'(x)$ là độ dốc của $f$ tại điểm đó: dương nghĩa là $f$ tăng khi $x$ tăng, âm nghĩa là nó giảm. Vì vậy bước một bước nhỏ *ngược* chiều độ dốc luôn làm giảm $f$ (với bước đủ nhỏ) &mdash; đó là toàn bộ ý tưởng đằng sau hạ gradient:

$$x \leftarrow x - \eta\, f'(x)$$

trong đó $\eta$ (gọi là **tốc độ học — learning rate**) điều khiển bước đi lớn đến đâu. Lặp lại cập nhật này và $x$ sẽ đi xuống dốc, dừng lại gần một cực tiểu, nơi $f'(x) \approx 0$.

Cho $f$ là một parabol. Bắt đầu từ xa cực tiểu, mỗi bước di chuyển ngược với độ dốc cục bộ. Các bước tự thu nhỏ dần khi tiến gần đáy, vì bản thân độ dốc $f'(x)=2(x-3)$ thu nhỏ khi $x \to 3$ &mdash; hạ gradient tự nhiên chậm lại gần cực tiểu. Với parabol, cực tiểu cục bộ mà hạ gradient tìm được cũng là cực tiểu toàn cục, trong trường hợp này là tại $x=3$.

In [ ]:
def f(x):
    return (x - 3) ** 2 + 1

def fprime(x):
    return 2 * (x - 3)

def gd_path(x0, lr, n_steps=10):
    x = x0
    path = [x]
    for _ in range(n_steps):
        x = x - lr * fprime(x)
        path.append(x)
    return np.array(path)

path = gd_path(x0=-4.0, lr=0.3)
xs = np.linspace(-5, 8, 200)

plt.plot(xs, f(xs), color='gray', linewidth=1.5, label='f(x) = (x-3)^2 + 1')
plt.plot(path, f(path), 'o-', color='C1', markersize=5, linewidth=1.2, label='gradient descent steps')
plt.scatter([path[0]], [f(path[0])], color='green', s=70, zorder=3, label='start')
plt.scatter([path[-1]], [f(path[-1])], color='red', s=70, zorder=3, label='after 10 steps')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Gradient descent on a 1D parabola')
plt.legend(fontsize=8)
plt.show()

Ảnh động này cho thấy cùng lần chạy đó từng bước một: đường tiếp tuyến nét đứt làm hiện rõ độ dốc tại điểm hiện tại, và mũi tên là bước đi ngược với nó &mdash; đúng quy tắc cập nhật ở trên, từng khung hình một.

<img src="../animations/gradient_descent_1d_demo.gif" alt="Ảnh động hạ gradient trên một parabol 1D: đường tiếp tuyến nét đứt cho thấy độ dốc tại điểm hiện tại và mũi tên cho thấy bước đi ngược với nó, hội tụ về cực tiểu" style="display: block; margin: 0 auto; max-width: 520px;">

Kích thước bước $\eta$ quan trọng không kém hướng đi: quá nhỏ thì tiến triển bò chậm; quá lớn thì các bước liên tiếp vượt quá đích ngày càng nhiều, phân kỳ thay vì hội tụ. Chọn một tốc độ học tốt là một trong những lựa chọn thực tế quan trọng nhất khi huấn luyện bất kỳ mô hình nào &mdash; một chủ đề mà Bài 33 sẽ quay lại chi tiết.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
settings = [(0.05, 'too small: slow'), (0.3, 'good: fast, settles'), (1.05, 'too large: diverges')]
for ax, (lr, label) in zip(axes, settings):
    path = gd_path(x0=-4.0, lr=lr, n_steps=10)
    ax.plot(xs, f(xs), color='gray', linewidth=1.2)
    ax.plot(path, f(path), 'o-', color='C1', markersize=4, linewidth=1)
    ax.scatter([path[0]], [f(path[0])], color='green', s=50, zorder=3)
    ax.scatter([path[-1]], [f(path[-1])], color='red', s=50, zorder=3)
    ax.set_title(f'lr={lr}\n({label})', fontsize=10)
    ax.set_xlabel('x')
axes[0].set_ylabel('f(x)')
plt.tight_layout()
plt.show()

## Một thư viện mới, và một cấu trúc dữ liệu mới: PyTorch và tensor

Mọi bài từ đầu đến giờ đều import `cv2` cho các thao tác ảnh. Bài này thay vào đó import `torch` &mdash; lần xuất hiện đầu tiên của PyTorch trong khóa học, và từ đây trở đi hầu hết các bài sẽ dựa vào nó thay vì OpenCV. Đó là vì xây dựng và huấn luyện mạng nơ-ron cần hai thứ mà OpenCV chưa bao giờ được thiết kế cho: vi phân tự động (automatic differentiation — tính gradient của các hàm phức tạp tùy ý, sẽ dùng bên dưới) và tăng tốc GPU cho các phép nhân ma trận lớn mà mạng nơ-ron được xây dựng từ đó.

Cùng với PyTorch là một từ mới: **tensor**. Tensor chỉ là thuật ngữ chung cho một lưới số có số chiều bất kỳ: một vô hướng là tensor 0 chiều, một vector là 1 chiều, một ma trận là 2 chiều, và một chồng ma trận &mdash; chẳng hạn một lô (batch) ảnh &mdash; là tensor 4 chiều. Mọi mảng đã dùng từ đầu khóa học &mdash; mảng NumPy, ảnh, ma trận trọng số &mdash; thực ra đều là tensor; `torch.Tensor` của PyTorch đơn giản là `ndarray` của NumPy được gắn thêm hai thứ: nó có thể theo dõi một `.grad` cho vi phân tự động, và nó có thể nằm trên GPU thay vì CPU để tính toán song song nhanh. `.shape` của một tensor cho biết đúng những gì `.shape` của NumPy cho biết với cùng mảng đó &mdash; hai thư viện gần nhau đến mức chuyển đổi giữa chúng (`torch.tensor(numpy_array)`, `tensor.numpy()`) về cơ bản là miễn phí.

Tensor ảnh mang theo hai quy ước đáng nêu ra ngay từ đầu:

- **Kênh đứng trước (channel-first).** PyTorch lưu ảnh màu dưới dạng `(channel, height, width)`, ngược với `(height, width, channel)` của NumPy/OpenCV ở Bài 1 &mdash; chuyển đổi giữa chúng nghĩa là di chuyển trục kênh, không chỉ đổi hình dạng: `image.transpose(2, 0, 1)` cho mảng NumPy (HWC&rarr;CHW), hoặc `tensor.permute(2, 0, 1)` cho tensor.
- **Chiều lô bắt buộc.** Các lớp và hàm của PyTorch hầu như luôn yêu cầu trục lô đứng đầu, ngay cả khi chỉ có một mẫu: một ảnh đơn lẻ đi vào với hình dạng `(1, channel, height, width)`, không phải `(channel, height, width)`. Quên điều này là nguồn gốc phổ biến của lỗi không khớp hình dạng; `tensor.unsqueeze(0)` chèn một trục lô kích thước 1 ở đầu, và `tensor.squeeze(0)` loại bỏ nó khi bạn quay lại xử lý một kết quả đơn lẻ.

## Một bài toán phân loại nhị phân

Ta dùng lại cùng tập dữ liệu hai lớp từ Bài 30, để hướng học được bên dưới có thể so sánh trực tiếp với hướng chọn thủ công.

In [ ]:
rng = np.random.default_rng(1)
class_a = rng.normal(loc=[-2, -1], scale=0.8, size=(60, 2))
class_b = rng.normal(loc=[2, 1.5], scale=0.8, size=(60, 2))
X = np.vstack([class_a, class_b])
y = np.concatenate([np.zeros(60), np.ones(60)])  # class A = 0, class B = 1

plt.scatter(*class_a.T, s=15, label='class A (y=0)')
plt.scatter(*class_b.T, s=15, label='class B (y=1)')
plt.legend(fontsize=8)
plt.gca().set_aspect('equal')
plt.title('Same two-class dataset as Lesson 30')
plt.show()

## Từ điểm số thô đến xác suất

Nhớ lại phép chiếu ở Bài 30: một nơ-ron đơn lẻ tính một điểm số thô $z = w^\top x + b$ từ đầu vào $x$, rồi phân loại theo dấu của điểm số đó. Mục tiêu của huấn luyện là tìm $w$ và $b$ làm cho $z$ dương với mọi điểm thuộc lớp B và âm với mọi điểm thuộc lớp A &mdash; nhưng thay vì chọn thủ công như Bài 30, ta muốn hạ gradient tự động tìm ra chúng.

Vấn đề là độ chính xác phân loại (đúng hoặc sai) phẳng gần như ở mọi nơi và nhảy vọt không liên tục tại ranh giới quyết định &mdash; nó không có gradient hữu ích nào để đi theo, nên hạ gradient không có gì để leo xuống. Cách khắc phục là thôi đòi hỏi một câu trả lời có/không cứng nhắc, và thay vào đó nén điểm số thô qua hàm **sigmoid**, ánh xạ bất kỳ số thực nào thành một giá trị mượt nằm giữa 0 và 1 &mdash; được hiểu là "mô hình tin chắc đến đâu rằng điểm này thuộc lớp B?"

$$\sigma(z) = \frac{1}{1+e^{-z}}$$

$z$ dương lớn ánh xạ gần 1, $z$ âm lớn ánh xạ gần 0, và $z=0$ ánh xạ chính xác thành 0.5, ngay trên lằn ranh.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

zs = np.linspace(-6, 6, 200)
plt.plot(zs, sigmoid(zs))
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.axvline(0, color='gray', linestyle='--', linewidth=1)
plt.xlabel('z = w.x + b')
plt.ylabel('sigma(z)')
plt.title('The sigmoid activation')
plt.show()

## Entropy chéo nhị phân: biến xác suất thành hàm mất mát huấn luyện được

Riêng một xác suất thì chưa phải là thứ để tối ưu. Huấn luyện cần một con số duy nhất &mdash; gọi là **sai số (error)**, hay **mất mát (loss)** &mdash; nhỏ khi các xác suất của mô hình tốt và lớn khi chúng tệ, lấy trung bình trên toàn bộ tập dữ liệu. **Entropy chéo nhị phân (binary cross-entropy)** xây dựng con số đó bằng cách phạt mỗi dự đoán theo mức độ tự tin của nó, và việc sự tự tin đó có chính đáng hay không: nếu nhãn thực là 1, mức phạt là $-\log(p)$ (gần 0 khi $p$ gần 1, bùng nổ về vô cùng khi $p\to 0$ &mdash; sai một cách tự tin bị phạt nặng); nếu nhãn thực là 0, nó là ảnh đối xứng, $-\log(1-p)$.

In [ ]:
p_range = np.linspace(0.001, 0.999, 200)
loss_if_true_1 = -np.log(p_range)
loss_if_true_0 = -np.log(1 - p_range)

plt.plot(p_range, loss_if_true_1, label='true label = 1:  -log(p)')
plt.plot(p_range, loss_if_true_0, label='true label = 0:  -log(1-p)')
plt.xlabel('predicted probability p')
plt.ylabel('per-example loss')
plt.legend(fontsize=8)
plt.title('Binary cross-entropy: the penalty for one prediction')
plt.show()

Lấy trung bình mức phạt trên mỗi mẫu đó trên toàn bộ tập dữ liệu cho ra hàm mất mát đầy đủ:

$$L = -\frac{1}{n}\sum_i \big[y_i \log \sigma(z_i) + (1-y_i)\log(1-\sigma(z_i))\big]$$

Đây chính xác là hai đường cong ở trên, được chọn bởi $y_i$ (chỉ một trong hai số hạng khác không, vì $y_i$ là 0 hoặc 1) và lấy trung bình trên mọi điểm. Vì cả sigmoid lẫn logarit đều trơn, $L$ có gradient xác định rõ ở mọi nơi &mdash; hàm mất mát giảm mượt mà khi dự đoán tiến gần tới đúng, thay vì chỉ thay đổi tại ranh giới quyết định.

## Tính gradient: áp dụng quy tắc chuỗi

Gradient được tìm bằng cách áp dụng quy tắc chuỗi (chain rule) trong giải tích cho hàm hợp này: $L$ phụ thuộc vào $\sigma(z)$, vốn phụ thuộc vào $z=w^\top x + b$, vốn phụ thuộc vào $w$ và $b$. Đi ngược qua chuỗi đó cho ra một kết quả gọn gàng đáng kinh ngạc (bỏ qua phần đại số, vốn đơn giản nhưng tẻ nhạt):

$$\frac{\partial L}{\partial z_i} = \sigma(z_i) - y_i, \qquad \frac{\partial L}{\partial w} = \frac{1}{n}X^\top(\sigma(z)-y), \qquad \frac{\partial L}{\partial b} = \frac{1}{n}\sum_i(\sigma(z_i)-y_i)$$

Nói cách khác: gradient chỉ là *sai số dự đoán*, được lấy trung bình (đánh trọng số theo $x$ với gradient của $w$). Các dự đoán sai nghiêm trọng đẩy trọng số mạnh; các dự đoán đúng hầu như không làm chúng di chuyển. Với chỉ một lớp ở đây, chuỗi này chỉ có một mắt xích; Bài 32 thêm mắt xích thứ hai, và đó là lúc việc áp dụng quy tắc chuỗi *lặp lại, từng lớp một* xứng đáng với cái tên **lan truyền ngược (backpropagation)**.

In [ ]:
def forward(X, w, b):
    z = X @ w + b
    return sigmoid(z), z

def bce_loss(p, y, eps=1e-9):
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def backward(X, p, y):
    grad_z = (p - y) / len(y)
    grad_w = X.T @ grad_z
    grad_b = grad_z.sum()
    return grad_w, grad_b

### Kiểm tra: kiểm tra gradient

Trước khi tin vào bất kỳ gradient nào tự suy ra, thông lệ tiêu chuẩn là kiểm tra nó bằng số: nhiễu loạn mỗi tham số một lượng rất nhỏ và xem hàm mất mát thực sự thay đổi bao nhiêu, rồi so sánh với công thức giải tích.

In [ ]:
w_test, b_test = np.array([0.5, -0.3]), 0.2
p_test, _ = forward(X, w_test, b_test)
grad_w_analytic, grad_b_analytic = backward(X, p_test, y)

def loss_at(w, b):
    p, _ = forward(X, w, b)
    return bce_loss(p, y)

eps = 1e-5
grad_w_numeric = np.zeros(2)
for i in range(2):
    w_plus, w_minus = w_test.copy(), w_test.copy()
    w_plus[i] += eps
    w_minus[i] -= eps
    grad_w_numeric[i] = (loss_at(w_plus, b_test) - loss_at(w_minus, b_test)) / (2 * eps)
grad_b_numeric = (loss_at(w_test, b_test + eps) - loss_at(w_test, b_test - eps)) / (2 * eps)

print(f'analytic grad_w: {grad_w_analytic}, numeric: {grad_w_numeric}')
print(f'analytic grad_b: {grad_b_analytic:.6f}, numeric: {grad_b_numeric:.6f}')
print(f'max discrepancy: {max(np.abs(grad_w_analytic - grad_w_numeric).max(), abs(grad_b_analytic - grad_b_numeric)):.2e}')

### Kiểm tra: autograd của PyTorch

Như một phép kiểm tra độc lập thứ hai, ta để PyTorch tự động tính cùng gradient đó qua `.backward()`, dùng hàm có sẵn `binary_cross_entropy_with_logits` (kết hợp sigmoid và hàm mất mát trong một hàm ổn định về mặt số).

In [ ]:
X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)
w_t = torch.tensor(w_test, dtype=torch.float64, requires_grad=True)
b_t = torch.tensor(b_test, dtype=torch.float64, requires_grad=True)

z_t = X_t @ w_t + b_t
loss_t = torch.nn.functional.binary_cross_entropy_with_logits(z_t, y_t)
loss_t.backward()

print(f'our analytic grad_w: {grad_w_analytic}')
print(f'torch autograd grad_w: {w_t.grad.numpy()}')
print(f'max discrepancy: {np.abs(grad_w_analytic - w_t.grad.numpy()).max():.2e}')

Hai phép kiểm tra độc lập &mdash; sai phân hữu hạn bằng số và vi phân tự động của PyTorch &mdash; đều khớp với công thức tự suy ra tới nhiều chữ số thập phân. Kiểu kiểm tra kép này là thông lệ tiêu chuẩn mỗi khi bạn tự suy ra một gradient.

## Huấn luyện: vòng lặp đầy đủ

Cùng ý tưởng đó mở rộng trực tiếp sang nhiều tham số cùng lúc: thay vì một độ dốc đơn lẻ $f'(x)$, **gradient** là một vector các đạo hàm riêng, mỗi tham số một đạo hàm &mdash; chính xác là $\partial L/\partial w$ và $\partial L/\partial b$ đã suy ra ở trên. Quy tắc cập nhật giống hệt về tinh thần: $w \leftarrow w - \eta\,\partial L/\partial w$, $b \leftarrow b - \eta\,\partial L/\partial b$. Khởi tạo $w, b$ ngẫu nhiên (lần này không dùng mẹo suy nghiệm thông minh nào), rồi lặp lại: lượt xuôi, tính hàm mất mát, lượt ngược, bước một bước nhỏ ngược chiều gradient.

Mỗi bước bên dưới dùng gradient tính trên *toàn bộ* tập huấn luyện cùng lúc (`n=120` mẫu) &mdash; đôi khi được gọi là hạ gradient toàn lô (full-batch gradient descent), để phân biệt với **hạ gradient ngẫu nhiên (stochastic gradient descent — SGD)** theo lô nhỏ (mini-batch) mà huấn luyện thực tế dùng, sẽ được Bài 33 trình bày chi tiết.

### Quan sát vòng lặp chạy, từng giai đoạn một

Ảnh động bên dưới chạy qua đúng công thức này &mdash; lượt xuôi, hàm mất mát, lượt ngược, cập nhật &mdash; từng giai đoạn có nhãn, trên một ví dụ nhỏ duy nhất: $x=(2, 1)$, $y=1$, bắt đầu từ $w=(-0.5, 0.3)$, $b=0$, với $\eta=0.5$. Khác với tập dữ liệu 120 điểm được huấn luyện bên dưới, các con số này đủ đơn giản để tính tay.

<img src="../animations/training_loop_demo.gif" alt="Ảnh động vòng lặp huấn luyện một nơ-ron: đồ thị tính toán với các khối w.x+b, sigmoid và hàm mất mát, luân phiên qua các giai đoạn lượt xuôi, hàm mất mát, lượt ngược và cập nhật kèm thanh tiến trình, trên một ví dụ tính tay ban đầu bị phân loại sai và hội tụ qua 3 lần cập nhật liên tiếp" style="display: block; margin: 0 auto; max-width: 760px;">

**Tự tính tay bước đầu tiên** là một cách tốt để xác nhận bạn đã hiểu các công thức ở trên trước khi tin dùng chúng trên một tập dữ liệu đầy đủ:

$$z = w^\top x + b = (-0.5)(2) + (0.3)(1) + 0 = -0.7$$
$$p = \sigma(z) = \sigma(-0.7) \approx 0.332$$
$$L = -\log(p) \approx 1.103 \quad\text{(since } y=1\text{, only the } -\log(p) \text{ term applies)}$$
$$\frac{\partial L}{\partial z} = p - y \approx 0.332 - 1 = -0.668$$
$$\frac{\partial L}{\partial w_1} = x_1 \cdot \frac{\partial L}{\partial z} \approx -1.336, \qquad \frac{\partial L}{\partial w_2} = x_2 \cdot \frac{\partial L}{\partial z} \approx -0.668, \qquad \frac{\partial L}{\partial b} \approx -0.668$$
$$w_1 \leftarrow -0.5 - 0.5(-1.336) \approx 0.168, \qquad w_2 \leftarrow 0.3 - 0.5(-0.668) \approx 0.634, \qquad b \leftarrow 0 - 0.5(-0.668) \approx 0.334$$

Đó chính xác là điều ảnh động cho thấy sau lần cập nhật đầu tiên &mdash; mô hình ban đầu sai một cách tự tin ($p=0.33$ khi $y=1$), và chỉ một bước đã lật dự đoán thành đúng ($p=0.79$) và giảm hàm mất mát khoảng ba phần tư (từ $1.10$ xuống $0.24$). Các con số sau này được để lại như bài tập cho người đọc (thay $w, b$ đã cập nhật trở lại vào cùng các công thức ở trên).

#### Quay lại tập dữ liệu thật

Khi cơ chế đã được xác nhận bằng tay, phần còn lại của mục này quay lại huấn luyện trên toàn bộ tập dữ liệu hai cụm 120 điểm từ Bài 30 &mdash; cùng các công thức, chỉ là áp dụng cho mọi mẫu cùng lúc thay vì một mẫu.

In [ ]:
def train(X, y, n_epochs=500, lr=0.5, seed=0):
    rng_local = np.random.default_rng(seed)
    w = rng_local.normal(size=X.shape[1]) * 0.1
    b = 0.0
    losses = []
    for _ in range(n_epochs):
        p, _ = forward(X, w, b)
        losses.append(bce_loss(p, y))
        grad_w, grad_b = backward(X, p, y)
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b, losses

w_learned, b_learned, losses = train(X, y)
final_pred = (sigmoid(X @ w_learned + b_learned) > 0.5).astype(float)
print(f'learned w = {np.round(w_learned, 3)}, b = {b_learned:.3f}')
print(f'final accuracy: {(final_pred == y).mean():.1%}')

plt.plot(losses)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss')
plt.show()

### So sánh với hướng chọn thủ công ở Bài 30

In [ ]:
w_handpicked = class_b.mean(axis=0) - class_a.mean(axis=0)
w_handpicked /= np.linalg.norm(w_handpicked)
w_learned_normalized = w_learned / np.linalg.norm(w_learned)

cos_angle = w_handpicked @ w_learned_normalized
print(f'hand-picked direction (Lesson 30): {np.round(w_handpicked, 3)}')
print(f'learned direction (this lesson):   {np.round(w_learned_normalized, 3)}')
print(f'angle between them: {np.degrees(np.arccos(np.clip(cos_angle, -1, 1))):.1f} degrees')

Hạ gradient, bắt đầu từ không gì khác ngoài nhiễu ngẫu nhiên, tái khám phá về cơ bản cùng hướng mà một người đã chọn bằng cách suy luận về các trung bình lớp &mdash; đáng yên tâm, nhưng cũng là lời báo trước về giới hạn của một nơ-ron đơn lẻ: nó chỉ có thể *tái khám phá* những gì một phép chiếu tuyến tính đơn lẻ có khả năng làm.

## Huấn luyện trên bài toán không giải được

Bài 30 đã chỉ ra rằng *không* phép chiếu tuyến tính nào tách được một vành khuyên khỏi hình đĩa mà nó bao quanh &mdash; tìm kiếm vét cạn tốt nhất trên các hướng và ngưỡng chỉ đạt độ chính xác 74%. Điều gì xảy ra khi hạ gradient, thay vì tìm kiếm vét cạn, cố gắng giải cùng bài toán đó?

In [ ]:
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))
X_ring = np.vstack([inner, outer])
y_ring = np.concatenate([np.zeros(60), np.ones(60)])

w_ring, b_ring, losses_ring = train(X_ring, y_ring, n_epochs=2000)
pred_ring = (sigmoid(X_ring @ w_ring + b_ring) > 0.5).astype(float)
print(f'learned w = {np.round(w_ring, 4)} (norm={np.linalg.norm(w_ring):.4f}), b = {b_ring:.4f}')
print(f'trained single-neuron accuracy: {(pred_ring == y_ring).mean():.1%}')
print(f'(Lesson 30\'s exhaustive-search ceiling for ANY linear separator was 74%)')

Ảnh động này cho thấy quá trình huấn luyện trên tập dữ liệu này: hãy quan sát hàm mất mát giảm mạnh trong vài epoch đầu rồi phẳng ra gần như ngay lập tức, hầu như không di chuyển trong các epoch còn lại, trong khi ranh giới quyết định dừng lại ở một vị trí gần như tùy ý.

<img src="../animations/gradient_descent_unsolvable_demo.gif" alt="Ảnh động hạ gradient huấn luyện một bộ phân loại một nơ-ron trên tập dữ liệu vành-khuyên-và-đĩa, với ranh giới quyết định và đường cong hàm mất mát tiến triển cùng nhau và hội tụ về độ chính xác gần mức đoán ngẫu nhiên, tệ hơn mức trần tìm kiếm vét cạn của Bài 30" style="display: block; margin: 0 auto; max-width: 720px;">

Hạ gradient ở đây thực ra còn làm *tệ hơn* tìm kiếm vét cạn &mdash; nó hội tụ về một vector trọng số gần 0 và độ chính xác ngang mức đoán ngẫu nhiên, thay vì tìm ra cung nhỏ, lệch một bên đã giúp tìm kiếm vét cạn vớt vát được 74%. Tập dữ liệu (gần như) đối xứng quanh gốc tọa độ, nên lực kéo gradient *trung bình* từ tất cả các điểm huấn luyện gần như triệt tiêu theo mọi hướng, và bộ tối ưu dừng lại gần $w\approx 0$ thay vì săn tìm một nghiệm ngóc ngách bất đối xứng. Kiểu thất bại khác, nhưng cùng một sự thật bên dưới: một phép chiếu tuyến tính đơn lẻ không thể giải bài toán này, bất kể nó được tìm ra bằng cách nào. Bài 32 khắc phục điều này &mdash; không phải bằng một bộ tối ưu thông minh hơn, mà bằng cách cho mô hình nhiều hơn một phép chiếu để làm việc.

### Bài tập

1. Tăng `lr` trong `train` vượt xa một giá trị hợp lý (ví dụ `lr=20`) trên tập dữ liệu hai cụm. Điều gì xảy ra với đường cong hàm mất mát, và điều này liên quan thế nào đến việc kích thước bước vượt quá độ cong của bề mặt hàm mất mát?
2. Huấn luyện lại trên tập dữ liệu hai cụm với vài seed ngẫu nhiên khác nhau. Hướng học được có luôn kết thúc gần hướng chọn thủ công ở Bài 30 không, hay nó dao động nhiều? Điều đó gợi ý gì về số lượng nghiệm tốt tồn tại cho một tập dữ liệu tách biệt tốt?
3. Làm cho vành ngoài bất đối xứng thay vì một vòng tròn đầy đủ (ví dụ `theta_out = rng.uniform(0, np.pi, 60)`, để lớp ngoài chỉ chiếm một nửa vành). Huấn luyện lại nơ-ron đơn lẻ. Sự phá vỡ đối xứng thực sự này có làm thay đổi đáng kể độ chính xác cuối cùng so với kết quả ~50% ngang mức đoán ngẫu nhiên trên vành đầy đủ không, và điều đó có ủng hộ lời giải thích ở trên (rằng lực kéo đối xứng của vành *đầy đủ* lên gradient, chứ không phải một sự bất lực cơ bản nào trong việc học, là thứ khiến huấn luyện đình trệ gần $w \approx 0$) không? (Lưu ý: chỉ đơn giản tịnh tiến cả hai lớp cùng một độ lệch sẽ *không* phá vỡ được đối xứng này, vì số hạng độ lệch $b$ có thể hấp thụ miễn phí bất kỳ phép tịnh tiến chung nào.)